# Modeling

In [1]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from src.models.model_0 import TabularModel

In [2]:
batch_size = 32

pl.seed_everything(42)

Seed set to 42


42

## GPU

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA GeForce RTX 3070


## Data loading and verification

In [4]:
X_train = np.load('../data/processed/X_train.npy')
X_val   = np.load('../data/processed/X_val.npy')
X_test  = np.load('../data/processed/X_test.npy')

y_train = np.load('../data/processed/y_train.npy')
y_val   = np.load('../data/processed/y_val.npy')
y_test  = np.load('../data/processed/y_test.npy')

In [5]:
for name, arr in [
    ("X_train", X_train), ("X_val", X_val), ("X_test", X_test),
    ("y_train", y_train), ("y_val", y_val), ("y_test", y_test)
]:
    print(f"{name} shape: {arr.shape}, dtype: {arr.dtype}")

X_train shape: (7000, 15), dtype: float64
X_val shape: (1500, 15), dtype: float64
X_test shape: (1500, 15), dtype: float64
y_train shape: (7000,), dtype: int64
y_val shape: (1500,), dtype: int64
y_test shape: (1500,), dtype: int64


In [6]:
# Inputs (float32)
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_val_tensor   = torch.from_numpy(X_val.astype(np.float32))
X_test_tensor  = torch.from_numpy(X_test.astype(np.float32))

# Targets (float32 for BCEWithLogitsLoss)
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_val_tensor   = torch.from_numpy(y_val.astype(np.float32))
y_test_tensor  = torch.from_numpy(y_test.astype(np.float32))

In [7]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

## Architecture

In [8]:
input_dim = X_train.shape[1]
output_dim = 1

In [9]:
model = TabularModel(input_dim=input_dim)

## Training

In [10]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,
    mode="min"            # "min" for loss, "max" for accuracy
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",             # "min" for loss, "max" for accuracy
    save_top_k=1,
    dirpath="../output/models/checkpoints/",
    filename="model-{epoch:02d}-{val_loss:.4f}"
)

#logger = TensorBoardLogger("tb_logs", name="model_0")

trainer = pl.Trainer(
    #logger=logger,
    max_epochs=100,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    callbacks=[early_stop_callback, checkpoint_callback]
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [11]:
trainer.fit(model, train_loader, val_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 3070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
C:\Users\danilo\DataspellProjects\pytorch-tutorial2\venv\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory C:\Users\danilo\DataspellProjects\pytorch-tutorial2\output\models\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type              | Params | Mode 
------------------------------------------------------
0 | model   | Sequential        | 1.2 K  | train
1 | loss_fn | BCEWithLogitsLoss | 0      | train
------------------------------------------------------
1.2 K     Trainable params
0         Non-trainable params
1.2 K     Total params
0.0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\danilo\DataspellProjects\pytorch-tutorial2\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
C:\Users\danilo\DataspellProjects\pytorch-tutorial2\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]